# NutriMatch: Advanced Food Recommendation Model
This notebook implements the advanced Deep Learning model for the NutriMatch system.
**Improvements Included:**
1. **Multi-Head Architecture**: Takes both macronutrient and allergy features, predicting match score and allergy risk.
2. **Asymmetric Custom Loss**: Penalizes false negatives on allergy predictions heavily.
3. **Train-Validation Split**: Evaluates precision, recall, accuracy, and MAE on a separate validation set using custom training loop.
4. **Rule-Based Guardrails**: Enforces absolute safety at the inference stage before providing final recommendations.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from datetime import datetime
import os

# Set seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## 1. Load Data & Parse Features

In [2]:
# Load datasets
food_df = pd.read_csv('Data/train_ready_dataset.csv')
user_df = pd.read_csv('Data/user_profile_features_schema.csv')

print(f"Original Food data shape: {food_df.shape}")
print(f"Original User data shape: {user_df.shape}")

# Use subset for faster iteration
food_subset = food_df.copy()
user_subset = user_df.copy()

# Parse user allergy vector from string "0,0,0,0,0,0,0,0" to 8 separate integer columns
allergy_cols_user = [f'user_allergy_{i}' for i in range(8)]
user_allergies = user_subset['allergy_vector'].str.split(',', expand=True).astype(np.float32)
user_allergies.columns = allergy_cols_user

# SYNTHETIC ALLERGY INJECTION
# Inject random allergies (e.g., 20% probability for each allergy type per user) to test the Recall metric.
np.random.seed(42)
random_allergies = np.random.choice([0.0, 1.0], size=user_allergies.shape, p=[0.8, 0.2])
user_allergies = pd.DataFrame(np.maximum(user_allergies.values, random_allergies), columns=allergy_cols_user)

user_subset = pd.concat([user_subset, user_allergies], axis=1)

# Map food allergens to 8 features (assuming standard mapping based on user schema)
# 1. gluten, 2. dairy, 3. nuts, 4. peanut, 5. seafood, 6. egg, 7. soy, 8. celery
allergy_cols_food = [
    'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_peanut', 
    'contains_seafood', 'contains_egg', 'contains_soy', 'contains_celery'
]
# Convert boolean/string to float32
for col in allergy_cols_food:
    food_subset[col] = food_subset[col].astype(bool).astype(np.float32)

Original Food data shape: (1332, 30)
Original User data shape: (100, 16)


## 2. Data Preprocessing & Synthetic Pairs (Train/Val Split)
Generate synthetic pairs and true labels for Match Score (Regression) and Allergy Danger (Classification).

In [3]:
# Define feature columns
user_macro_cols = ['target_calorie', 'protein_target_g', 'fat_target_g', 'carb_target_g']
food_macro_cols = ['calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g']

user_subset[user_macro_cols] = user_subset[user_macro_cols].fillna(0)
food_subset[food_macro_cols] = food_subset[food_macro_cols].fillna(0)

# Cartesian product for pairs
user_subset['dummy_key'] = 1
food_subset['dummy_key'] = 1
pairs = pd.merge(user_subset, food_subset, on='dummy_key').drop('dummy_key', axis=1)

# Synthetic Label 1: Match Score (Regression)
calorie_diff = np.abs((pairs['target_calorie'] / 3) - pairs['calories_100g'])
pairs['match_score'] = np.exp(-calorie_diff / 500).astype(np.float32)

# Synthetic Label 2: Allergy Risk (Classification)
# 1 if ANY user allergy overlaps with food allergy, else 0
user_all_arr = pairs[allergy_cols_user].values
food_all_arr = pairs[allergy_cols_food].values
pairs['is_allergic'] = np.any(np.logical_and(user_all_arr == 1, food_all_arr == 1), axis=1).astype(np.float32)

# Prepare inputs
X_user_macro = pairs[user_macro_cols].values.astype(np.float32)
X_food_macro = pairs[food_macro_cols].values.astype(np.float32)
X_user_allergy = pairs[allergy_cols_user].values.astype(np.float32)
X_food_allergy = pairs[allergy_cols_food].values.astype(np.float32)

y_match = pairs['match_score'].values
y_allergy = pairs['is_allergic'].values

# Normalization for Macros (Allergies are already 0/1)
user_max = np.max(X_user_macro, axis=0) + 1e-9
food_max = np.max(X_food_macro, axis=0) + 1e-9
X_user_macro = X_user_macro / user_max
X_food_macro = X_food_macro / food_max

# Shuffle and Split (80% Train, 20% Val)
total_samples = len(pairs)
indices = np.random.permutation(total_samples)
train_idx, val_idx = indices[:int(0.8 * total_samples)], indices[int(0.8 * total_samples):]

def create_dataset(idx):
    return tf.data.Dataset.from_tensor_slices((
        (X_user_macro[idx], X_user_allergy[idx], X_food_macro[idx], X_food_allergy[idx]),
        (y_match[idx], y_allergy[idx])
    )).batch(64)

train_dataset = create_dataset(train_idx)
val_dataset = create_dataset(val_idx)

print(f"Total pairs: {total_samples} (Train: {len(train_idx)}, Val: {len(val_idx)})")
print(f"Total allergic pairs in train: {np.sum(y_allergy[train_idx])}")

Total pairs: 133200 (Train: 106560, Val: 26640)
Total allergic pairs in train: 4436.0


## 3. Multi-Head Architecture & Custom Layer
Designing the Multi-Task Learning model.

In [4]:
# 1. Custom Layer
class InteractionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(InteractionLayer, self).__init__(**kwargs)
    def call(self, user_embed, food_embed):
        return tf.multiply(user_embed, food_embed)

# 2. Multi-Head Model Architecture
user_macro_in = tf.keras.Input(shape=(4,), name='user_macro')
user_allergy_in = tf.keras.Input(shape=(8,), name='user_allergy')
food_macro_in = tf.keras.Input(shape=(4,), name='food_macro')
food_allergy_in = tf.keras.Input(shape=(8,), name='food_allergy')

# Branch 1: Macronutrient Match
u_mac_dense = tf.keras.layers.Dense(16, activation='relu')(user_macro_in)
f_mac_dense = tf.keras.layers.Dense(16, activation='relu')(food_macro_in)
mac_interact = InteractionLayer()(u_mac_dense, f_mac_dense)
match_output = tf.keras.layers.Dense(1, activation='sigmoid', name='match_output')(mac_interact)

# Branch 2: Allergy Risk Classification
concat_allergy = tf.keras.layers.Concatenate()([user_allergy_in, food_allergy_in])
all_dense = tf.keras.layers.Dense(16, activation='relu')(concat_allergy)
allergy_output = tf.keras.layers.Dense(1, activation='sigmoid', name='allergy_output')(all_dense)

model = tf.keras.Model(
    inputs=[user_macro_in, user_allergy_in, food_macro_in, food_allergy_in], 
    outputs=[match_output, allergy_output]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_macro          │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ food_macro          │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_allergy        │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ food_allergy        │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │         80 │ user_macro[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         80 │ food_macro[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16)        │          0 │ user_allergy[0][… │
│ (Concatenate)       │                   │            │ food_allergy[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ interaction_layer   │ (None, 16)        │          0 │ dense[0][0],      │
│ (InteractionLayer)  │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16)        │        272 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ match_output        │ (None, 1)         │         17 │ interaction_laye… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ allergy_output      │ (None, 1)         │         17 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 466 (1.82 KB)

 Trainable params: 466 (1.82 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Asymmetric Custom Loss & TensorBoard Setup
`AsymmetricAllergyLoss` heavily penalizes false negatives (predicting safe when it's actually allergic).

In [5]:
# Custom Asymmetric Loss for Allergy
def asymmetric_allergy_loss(y_true, y_pred):
    y_true = tf.reshape(y_true, [-1, 1])
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    # Penalty: 100x larger if True=1 (Allergic) but Pred < 0.5 (Safe) -> False Negative
    y_true_bool = tf.cast(y_true, tf.bool)
    y_pred_safe = tf.cast(y_pred < 0.5, tf.bool)
    false_negative = tf.logical_and(y_true_bool, y_pred_safe)
    
    penalty = tf.where(false_negative, 100.0, 1.0)
    return tf.reduce_mean(bce * penalty)

logdir = os.path.join("logs", "multihead_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
summary_writer = tf.summary.create_file_writer(logdir)

## 5. Custom Train & Validation Loop

In [6]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
mse_loss_fn = tf.keras.losses.MeanSquaredError()

# Metrics
train_mae = tf.keras.metrics.MeanAbsoluteError()
val_mae = tf.keras.metrics.MeanAbsoluteError()
train_recall = tf.keras.metrics.Recall(thresholds=0.5)
val_recall = tf.keras.metrics.Recall(thresholds=0.5)

epochs = 10
for epoch in range(epochs):
    print(f"\n--- Epoch {epoch+1}/{epochs} ---")
    
    # ---------------- TRAINING ----------------
    for x_batch, y_batch in train_dataset:
        with tf.GradientTape() as tape:
            match_pred, allergy_pred = model(x_batch, training=True)
            loss_match = mse_loss_fn(tf.expand_dims(y_batch[0], -1), match_pred)
            loss_allergy = asymmetric_allergy_loss(y_batch[1], allergy_pred)
            total_loss = loss_match + loss_allergy
            
        grads = tape.gradient(total_loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        train_mae.update_state(tf.expand_dims(y_batch[0], -1), match_pred)
        train_recall.update_state(tf.expand_dims(y_batch[1], -1), allergy_pred)
        
    # ---------------- VALIDATION ----------------
    for x_val, y_val in val_dataset:
        val_match_pred, val_allergy_pred = model(x_val, training=False)
        val_mae.update_state(tf.expand_dims(y_val[0], -1), val_match_pred)
        val_recall.update_state(tf.expand_dims(y_val[1], -1), val_allergy_pred)
        
    # Results
    t_mae, t_rec = train_mae.result(), train_recall.result()
    v_mae, v_rec = val_mae.result(), val_recall.result()
    print(f"Train -> MAE: {t_mae:.4f} | Allergy Recall: {t_rec:.4f}")
    print(f"Val   -> MAE: {v_mae:.4f} | Allergy Recall: {v_rec:.4f}")
    
    # TensorBoard Log
    with summary_writer.as_default():
        tf.summary.scalar('Train_MAE', t_mae, step=epoch)
        tf.summary.scalar('Train_Recall', t_rec, step=epoch)
        tf.summary.scalar('Val_MAE', v_mae, step=epoch)
        tf.summary.scalar('Val_Recall', v_rec, step=epoch)
        
    # Reset states for next epoch
    train_mae.reset_state()
    val_mae.reset_state()
    train_recall.reset_state()
    val_recall.reset_state()


--- Epoch 1/10 ---
Train -> MAE: 0.0288 | Allergy Recall: 0.7015
Val   -> MAE: 0.0199 | Allergy Recall: 0.9724

--- Epoch 2/10 ---
Train -> MAE: 0.0210 | Allergy Recall: 0.9874
Val   -> MAE: 0.0197 | Allergy Recall: 0.9945

--- Epoch 3/10 ---
Train -> MAE: 0.0208 | Allergy Recall: 0.9964
Val   -> MAE: 0.0197 | Allergy Recall: 0.9945

--- Epoch 4/10 ---
Train -> MAE: 0.0197 | Allergy Recall: 0.9986
Val   -> MAE: 0.0179 | Allergy Recall: 0.9945

--- Epoch 5/10 ---
Train -> MAE: 0.0187 | Allergy Recall: 0.9989
Val   -> MAE: 0.0176 | Allergy Recall: 0.9945

--- Epoch 6/10 ---
Train -> MAE: 0.0186 | Allergy Recall: 0.9984
Val   -> MAE: 0.0176 | Allergy Recall: 0.9954

--- Epoch 7/10 ---
Train -> MAE: 0.0186 | Allergy Recall: 0.9995
Val   -> MAE: 0.0176 | Allergy Recall: 0.9945

--- Epoch 8/10 ---
Train -> MAE: 0.0186 | Allergy Recall: 0.9993
Val   -> MAE: 0.0176 | Allergy Recall: 0.9945

--- Epoch 9/10 ---
Train -> MAE: 0.0185 | Allergy Recall: 0.9998
Val   -> MAE: 0.0175 | Allergy Recall:

## 6. Save Model

In [9]:
model.save('nutrimatch_model.keras')
print("Model saved as 'nutrimatch_model.keras'")

Model saved as 'nutrimatch_model.keras'


## 7. Inference & Rule-Based Guardrails
During inference, even if AI says it's 99% safe (allergy risk 1%), the guardrail checks the absolute boolean values to ensure 100% safety.

In [8]:
# 7. Real Case Recommendation & Meal Planning
# Implementing Portion Sizing and Meal Time Distribution (Breakfast, Lunch, Dinner)

loaded_model = tf.keras.models.load_model(
    'nutrimatch_model.keras', 
    custom_objects={'InteractionLayer': InteractionLayer, 'asymmetric_allergy_loss': asymmetric_allergy_loss}
)

def get_meal_recommendations(meal_name, target_macros, user_allergy_list, top_k=2):
    # target_macros: [cal, prot, fat, carb]
    u_mac = np.array(target_macros, dtype=np.float32) / user_max
    u_all = np.array(user_allergy_list, dtype=np.float32)
    
    num_foods = len(food_subset)
    u_mac_batch = np.tile(u_mac, (num_foods, 1))
    u_all_batch = np.tile(u_all, (num_foods, 1))
    f_mac_batch = X_food_macro[:num_foods]
    f_all_batch = X_food_allergy[:num_foods]
    
    # Predict matching score using the AI model
    match_scores, allergy_risks = loaded_model.predict(
        [u_mac_batch, u_all_batch, f_mac_batch, f_all_batch], 
        batch_size=64, verbose=0
    )
    
    # Guardrail: Check for overlapping allergies
    has_allergy = np.any(np.logical_and(u_all_batch == 1, f_all_batch == 1), axis=1)
    
    results = []
    target_cal = target_macros[0]
    
    for i in range(num_foods):
        if not has_allergy[i]: # Guardrail blocks allergic foods
            cal_100g = food_subset.iloc[i]['calories_100g']
            if cal_100g > 0: 
                # Feature 1: Portion Sizing Calculation
                ideal_grams = (target_cal / cal_100g) * 100
                
                ideal_prot = (food_subset.iloc[i]['protein_100g'] / 100) * ideal_grams
                ideal_fat = (food_subset.iloc[i]['fat_100g'] / 100) * ideal_grams
                ideal_carb = (food_subset.iloc[i]['carbohydrate_100g'] / 100) * ideal_grams
                
                results.append({
                    'food_name': food_subset.iloc[i]['food_name'],
                    'ideal_grams': ideal_grams,
                    'ideal_prot': ideal_prot,
                    'ideal_fat': ideal_fat,
                    'ideal_carb': ideal_carb,
                    'match_score': match_scores[i][0]
                })
            
    # Sort by AI Match Score
    results = sorted(results, key=lambda x: x['match_score'], reverse=True)
    
    print(f"\n🍽️ === {meal_name} (Target: {target_cal:.0f} kcal) ===")
    if not results:
        print("   [!] Tidak ada makanan yang aman dari alergi Anda di database.")
        return
        
    for i, res in enumerate(results[:top_k]):
        print(f"Opsi {i+1}: {res['food_name'].title()}")
        print(f"   -> Porsi Ideal: {res['ideal_grams']:.0f} gram")
        print(f"   -> Gizi didapat: {res['ideal_prot']:.1f}g Prot | {res['ideal_fat']:.1f}g Lemak | {res['ideal_carb']:.1f}g Karbo")
        print(f"   -> AI Match Score: {res['match_score']*100:.2f}%")

def generate_daily_meal_plan(daily_macros, user_allergy_list):
    print(f"=== 📅 JADWAL MAKAN HARIAN ===")
    print(f"Target Total: {daily_macros[0]} Kalori")
    print(f"Alergi [Gluten, Dairy, Nuts, Peanut, Seafood, Egg, Soy, Celery]: {user_allergy_list}")
    
    # Feature 2: Meal Time Distribution
    # Membagi persentase kalori harian
    ratios = {'Sarapan Pagi (25%)': 0.25, 'Makan Siang (40%)': 0.40, 'Makan Malam (35%)': 0.35}
    
    for meal_name, ratio in ratios.items():
        # Skalakan seluruh target gizi berdasarkan persentase jadwal makan
        meal_macros = [m * ratio for m in daily_macros]
        get_meal_recommendations(meal_name, meal_macros, user_allergy_list, top_k=2)

# ---- TEST CASE 1: User sehat tanpa alergi ----
print("\n" + "="*50)
print("TEST CASE 1: User Biasa")
my_daily_macros_1 = [1800, 100, 50, 200]
my_allergies_1 = [0, 0, 0, 0, 0, 0, 0, 0] # Tidak ada alergi
generate_daily_meal_plan(my_daily_macros_1, my_allergies_1)

# ---- TEST CASE 2: User dengan pantangan ----
print("\n" + "="*50)
print("TEST CASE 2: User Alergi Seafood & Telur")
my_daily_macros_2 = [2200, 120, 60, 250]
my_allergies_2 = [0, 0, 0, 0, 1, 1, 0, 0] # Alergi Seafood & Telur
generate_daily_meal_plan(my_daily_macros_2, my_allergies_2)



TEST CASE 1: User Biasa
=== 📅 JADWAL MAKAN HARIAN ===
Target Total: 1800 Kalori
Alergi [Gluten, Dairy, Nuts, Peanut, Seafood, Egg, Soy, Celery]: [0, 0, 0, 0, 0, 0, 0, 0]

🍽️ === Sarapan Pagi (25%) (Target: 450 kcal) ===
Opsi 1: Minyak Zaitun
   -> Porsi Ideal: 51 gram
   -> Gizi didapat: 0.0g Prot | 50.9g Lemak | 0.0g Karbo
   -> AI Match Score: 98.00%
Opsi 2: Minyak Kedelai
   -> Porsi Ideal: 51 gram
   -> Gizi didapat: 0.0g Prot | 50.9g Lemak | 0.0g Karbo
   -> AI Match Score: 97.99%

🍽️ === Makan Siang (40%) (Target: 720 kcal) ===
Opsi 1: Minyak Zaitun
   -> Porsi Ideal: 81 gram
   -> Gizi didapat: 0.0g Prot | 81.4g Lemak | 0.0g Karbo
   -> AI Match Score: 98.14%
Opsi 2: Minyak Kedelai
   -> Porsi Ideal: 82 gram
   -> Gizi didapat: 0.0g Prot | 81.5g Lemak | 0.0g Karbo
   -> AI Match Score: 98.13%

🍽️ === Makan Malam (35%) (Target: 630 kcal) ===
Opsi 1: Minyak Zaitun
   -> Porsi Ideal: 71 gram
   -> Gizi didapat: 0.0g Prot | 71.3g Lemak | 0.0g Karbo
   -> AI Match Score: 98.09%
Opsi